## HW2

In [17]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models, datasets
from PIL import Image
import os
import pandas as pd
from tqdm.notebook import tqdm
import numpy as np
import itertools
import copy

# -- Конфигурация --
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Используемое устройство: {DEVICE}")

# Пути к данным
BASE_PATH = 'data/'
UNLABELED_PATH = os.path.join(BASE_PATH, 'train', 'unlabeled')
LABELED_PATH = os.path.join(BASE_PATH, 'train', 'labeled')
TEST_PATH = os.path.join(BASE_PATH, 'test')

# -- Гиперпараметры --
# Для Jigsaw может потребоваться больше эпох для сходимости
PRETRAIN_EPOCHS = 30
FINETUNE_EPOCHS = 30
BATCH_SIZE = 32 # Уменьшаем батч, так как модель больше
LEARNING_RATE_PRETRAIN = 1e-3
LEARNING_RATE_FINETUNE = 1e-4
GRID_SIZE = 3 # Сетка 3x3
PATCH_SIZE = 64 # Размер каждой части
N_PERMUTATIONS = 100 # Количество перестановок для предсказания

Используемое устройство: cuda


In [2]:
class RotationDataset(Dataset):
    """Custom Dataset for the rotation pretext task."""
    def __init__(self, image_dir, transform=None):
        self.image_dir = image_dir
        self.transform = transform
        self.image_files = [os.path.join(image_dir, f) for f in os.listdir(image_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_path = self.image_files[idx]
        image = Image.open(img_path).convert('RGB')

        # Apply the base transformations
        if self.transform:
            image = self.transform(image)

        # Apply a random rotation and create the label
        rotation_angle_idx = torch.randint(0, 4, (1,)).item() # 0: 0, 1: 90, 2: 180, 3: 270
        rotated_image = torch.rot90(image, k=rotation_angle_idx, dims=[1, 2])

        return rotated_image, rotation_angle_idx

# Define transformations for the pretext task
# Normalization values are standard for ImageNet, which work well in general
pretext_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Create Dataset and DataLoader
pretrain_dataset = RotationDataset(UNLABELED_PATH, transform=pretext_transforms)
pretrain_loader = DataLoader(pretrain_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)

print(f"Found {len(pretrain_dataset)} images for pre-training.")

Found 16611 images for pre-training.


In [3]:
# Load ResNet-18 and modify the classifier for the pretext task
pretext_model = models.resnet18()
num_features = pretext_model.fc.in_features
pretext_model.fc = nn.Linear(num_features, 4) # 4 classes for 4 rotation angles
pretext_model = pretext_model.to(DEVICE)

# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(pretext_model.parameters(), lr=LEARNING_RATE_PRETRAIN)

print("Starting self-supervised pre-training...")

for epoch in range(PRETRAIN_EPOCHS):
    pretext_model.train()
    running_loss = 0.0
    correct_predictions = 0
    total_predictions = 0

    progress_bar = tqdm(pretrain_loader, desc=f"Epoch {epoch+1}/{PRETRAIN_EPOCHS}")

    for images, labels in progress_bar:
        images, labels = images.to(DEVICE), labels.to(DEVICE)

        # Zero the parameter gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = pretext_model(images)
        loss = criterion(outputs, labels)

        # Backward pass and optimize
        loss.backward()
        optimizer.step()

        # Update statistics
        running_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total_predictions += labels.size(0)
        correct_predictions += (predicted == labels).sum().item()

        # Update progress bar
        progress_bar.set_postfix(loss=loss.item(), acc=f"{(correct_predictions/total_predictions):.2f}")

    epoch_loss = running_loss / len(pretrain_loader.dataset)
    epoch_acc = correct_predictions / total_predictions
    print(f"Pre-training Epoch {epoch+1}/{PRETRAIN_EPOCHS} - Loss: {epoch_loss:.4f}, Accuracy: {epoch_acc:.4f}")

print("Finished pre-training.")

# Save the backbone (feature extractor) weights
# We remove the final fully connected layer ('fc') before saving
backbone_weights = copy.deepcopy(pretext_model.state_dict())
backbone_weights.pop('fc.weight')
backbone_weights.pop('fc.bias')

torch.save(backbone_weights, 'resnet18_backbone_pretrained.pth')
print("Saved pre-trained backbone weights to 'resnet18_backbone_pretrained.pth'")

Starting self-supervised pre-training...


Epoch 1/50:   0%|          | 0/260 [00:00<?, ?it/s]

Pre-training Epoch 1/50 - Loss: 1.2621, Accuracy: 0.4327


Epoch 2/50:   0%|          | 0/260 [00:00<?, ?it/s]

Pre-training Epoch 2/50 - Loss: 1.1154, Accuracy: 0.5227


...

Pre-training Epoch 48/50 - Loss: 0.2276, Accuracy: 0.9169


Epoch 49/50:   0%|          | 0/260 [00:00<?, ?it/s]

Pre-training Epoch 49/50 - Loss: 0.2204, Accuracy: 0.9196


Epoch 50/50:   0%|          | 0/260 [00:00<?, ?it/s]

Pre-training Epoch 50/50 - Loss: 0.2155, Accuracy: 0.9238
Finished pre-training.
Saved pre-trained backbone weights to 'resnet18_backbone_pretrained.pth'


In [3]:
from torchvision import datasets

# Transformations for the downstream classification task
finetune_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Create dataset using ImageFolder
# Corrected line: Use 'datasets.ImageFolder' instead of 'models.datasets.ImageFolder'
full_labeled_dataset = datasets.ImageFolder(LABELED_PATH, transform=finetune_transforms)

# (Optional but Recommended) Split labeled data into train and validation sets
train_size = int(0.9 * len(full_labeled_dataset))
val_size = len(full_labeled_dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(full_labeled_dataset, [train_size, val_size])

# Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

# Get class names for submission file later
class_names = full_labeled_dataset.classes
print(f"Found {len(full_labeled_dataset)} labeled images in {len(class_names)} classes.")
print("Classes:", class_names)

Found 2075 labeled images in 10 classes.
Classes: ['butterfly', 'cat', 'chicken', 'cow', 'dog', 'elephant', 'horse', 'sheep', 'spider', 'squirrel']


In [7]:
# Initialize the final classification model
finetune_model = models.resnet18(num_classes=10)

# Load the pre-trained backbone weights
print("Loading pre-trained backbone weights...")
finetune_model.load_state_dict(torch.load('resnet18_backbone_pretrained.pth'), strict=False)
finetune_model = finetune_model.to(DEVICE)

# Define loss and optimizer for fine-tuning
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(finetune_model.parameters(), lr=LEARNING_RATE_FINETUNE)

best_val_acc = 0.0
best_model_wts = copy.deepcopy(finetune_model.state_dict())

print("Starting supervised fine-tuning...")

for epoch in range(FINETUNE_EPOCHS):
    # Training phase
    finetune_model.train()
    running_loss = 0.0
    correct_predictions = 0
    total_predictions = 0
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{FINETUNE_EPOCHS} (Train)")

    for images, labels in progress_bar:
        images, labels = images.to(DEVICE), labels.to(DEVICE)

        optimizer.zero_grad()
        outputs = finetune_model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total_predictions += labels.size(0)
        correct_predictions += (predicted == labels).sum().item()
        progress_bar.set_postfix(loss=loss.item())

    train_acc = correct_predictions / total_predictions

    # Validation phase
    finetune_model.eval()
    val_loss = 0.0
    val_correct = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs = finetune_model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs.data, 1)
            val_correct += (predicted == labels).sum().item()

    val_acc = val_correct / len(val_dataset)
    print(f"Epoch {epoch+1}/{FINETUNE_EPOCHS} - Train Acc: {train_acc:.4f}, Val Acc: {val_acc:.4f}")

    # Save the best model based on validation accuracy
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_model_wts = copy.deepcopy(finetune_model.state_dict())
        torch.save(best_model_wts, 'best_finetuned_model.pth')
        print(f"New best model saved with validation accuracy: {best_val_acc:.4f}")

print(f"Finished fine-tuning. Best Validation Accuracy: {best_val_acc:.4f}")

Loading pre-trained backbone weights...
Starting supervised fine-tuning...


Epoch 1/50 (Train):   0%|          | 0/59 [00:00<?, ?it/s]

Epoch 1/50 - Train Acc: 0.1784, Val Acc: 0.1971
New best model saved with validation accuracy: 0.1971


Epoch 2/50 (Train):   0%|          | 0/59 [00:00<?, ?it/s]

Epoch 2/50 - Train Acc: 0.2876, Val Acc: 0.3029
New best model saved with validation accuracy: 0.3029


...

Epoch 48/50 - Train Acc: 0.8972, Val Acc: 0.7067


Epoch 49/50 (Train):   0%|          | 0/59 [00:00<?, ?it/s]

Epoch 49/50 - Train Acc: 0.9009, Val Acc: 0.7212


Epoch 50/50 (Train):   0%|          | 0/59 [00:00<?, ?it/s]

Epoch 50/50 - Train Acc: 0.9095, Val Acc: 0.6779
Finished fine-tuning. Best Validation Accuracy: 0.7356


In [9]:
class TestDataset(Dataset):
    """Dataset for test images."""
    def __init__(self, test_dir, transform=None):
        self.test_dir = test_dir
        self.transform = transform
        self.image_files = sorted([f for f in os.listdir(test_dir) if f.endswith(('.png', '.jpg', '.jpeg'))])

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_path = os.path.join(self.test_dir, self.image_files[idx])
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, self.image_files[idx]

# Use validation transforms for the test set, but without random augmentations
test_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

test_dataset = TestDataset(TEST_PATH, transform=test_transforms)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

# Load the best model for inference
final_model = models.resnet18(num_classes=10)
final_model.load_state_dict(torch.load('best_finetuned_model.pth'))
final_model = final_model.to(DEVICE)
final_model.eval()

predictions = []
image_ids = []

print("Generating predictions on the test set...")
with torch.no_grad():
    for images, fnames in tqdm(test_loader):
        images = images.to(DEVICE)
        outputs = final_model(images)
        _, predicted_indices = torch.max(outputs, 1)

        predictions.extend([class_names[i] for i in predicted_indices.cpu().numpy()])
        image_ids.extend(fnames)

# Create submission DataFrame
submission_df = pd.DataFrame({
    'id': image_ids,
    'class': predictions
})

# Save to CSV
submission_df.to_csv('submission.csv', index=False)

print("Submission file 'submission.csv' created successfully!")
print(submission_df.head())

Generating predictions on the test set...


  0%|          | 0/65 [00:00<?, ?it/s]

Submission file 'submission.csv' created successfully!
         id     class
0     0.jpg       dog
1     1.jpg     horse
2    10.jpg     sheep
3   100.jpg    spider
4  1000.jpg  elephant


## Jigsaw

In [10]:
def generate_permutations(n_permutations, grid_size):
    """Генерирует и сохраняет фиксированный набор перестановок."""
    permutations_path = f'permutations_{n_permutations}.npy'
    if os.path.exists(permutations_path):
        print("Загрузка существующих перестановок...")
        perms = np.load(permutations_path)
        return perms

    print("Генерация новых перестановок...")
    n_patches = grid_size * grid_size
    permutations = []
    # Создаем все возможные перестановки
    all_perms = list(itertools.permutations(range(n_patches)))
    
    # Отбираем подмножество с максимальным расстоянием Хэмминга для разнообразия
    # (упрощенный вариант - просто случайный выбор)
    np.random.shuffle(all_perms)
    selected_perms = all_perms[:n_permutations]
    
    np.save(permutations_path, selected_perms)
    print(f"Сохранено {n_permutations} перестановок в {permutations_path}")
    return np.array(selected_perms)

# Генерируем или загружаем перестановки
permutations = generate_permutations(N_PERMUTATIONS, GRID_SIZE)

class JigsawDataset(Dataset):
    def __init__(self, image_dir, permutations, grid_size=3, patch_size=64, transform=None):
        self.image_dir = image_dir
        self.image_files = [os.path.join(image_dir, f) for f in os.listdir(image_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]
        self.permutations = permutations
        self.grid_size = grid_size
        self.patch_size = patch_size
        self.transform = transform

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_path = self.image_files[idx]
        image = Image.open(img_path).convert('RGB')
        
        # 1. Вырезаем центральный квадрат и делим на сетку
        img_size = self.grid_size * self.patch_size
        center_crop = transforms.CenterCrop(img_size)
        image = center_crop(image)
        
        patches = []
        for i in range(self.grid_size):
            for j in range(self.grid_size):
                box = (j * self.patch_size, i * self.patch_size, (j + 1) * self.patch_size, (i + 1) * self.patch_size)
                patch = image.crop(box)
                if self.transform:
                    patch = self.transform(patch)
                patches.append(patch)
        
        # 2. Выбираем случайную перестановку
        perm_index = np.random.randint(0, len(self.permutations))
        perm = self.permutations[perm_index]
        
        # 3. Перемешиваем части
        shuffled_patches = [patches[i] for i in perm]
        
        # Стекуем в один тензор
        shuffled_patches_tensor = torch.stack(shuffled_patches)
        
        return shuffled_patches_tensor, torch.tensor(perm_index, dtype=torch.long)

# Трансформации для каждой части изображения
pretext_transforms = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Создаем датасет и загрузчик
pretrain_dataset = JigsawDataset(UNLABELED_PATH, permutations, grid_size=GRID_SIZE, patch_size=PATCH_SIZE, transform=pretext_transforms)
pretrain_loader = DataLoader(pretrain_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)

print(f"Найдено {len(pretrain_dataset)} изображений для pre-training.")

Генерация новых перестановок...
Сохранено 100 перестановок в permutations_100.npy
Найдено 16611 изображений для pre-training.


In [11]:
class JigsawNet(nn.Module):
    def __init__(self, num_permutations):
        super(JigsawNet, self).__init__()
        # Загружаем ResNet-18, но без последнего слоя
        resnet_backbone = models.resnet18(weights=None) # weights=None, так как обучаем с нуля
        self.backbone = nn.Sequential(*list(resnet_backbone.children())[:-1])
        
        # Классификатор, который принимает объединенные признаки
        # 9 (частей) * 512 (признаков от ResNet18)
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear((GRID_SIZE**2) * 512, 1024),
            nn.ReLU(),
            nn.Linear(1024, num_permutations)
        )

    def forward(self, x):
        # x имеет размер (batch_size, 9, 3, patch_size, patch_size)
        batch_size, num_patches, c, h, w = x.shape
        
        # Объединяем batch_size и num_patches, чтобы обработать все части за один проход
        x = x.view(batch_size * num_patches, c, h, w)
        
        # Получаем признаки для каждой части
        features = self.backbone(x)
        
        # Возвращаем исходную размерность батча
        features = features.view(batch_size, num_patches, -1)
        
        # Подаем в классификатор
        output = self.classifier(features)
        return output

# Инициализируем модель
jigsaw_model = JigsawNet(num_permutations=N_PERMUTATIONS).to(DEVICE)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(jigsaw_model.parameters(), lr=LEARNING_RATE_PRETRAIN)

print("Начало self-supervised pre-training (Jigsaw)...")

for epoch in range(PRETRAIN_EPOCHS):
    jigsaw_model.train()
    running_loss = 0.0
    correct_predictions = 0
    total_predictions = 0

    progress_bar = tqdm(pretrain_loader, desc=f"Эпоха {epoch+1}/{PRETRAIN_EPOCHS}")

    for patches, labels in progress_bar:
        patches, labels = patches.to(DEVICE), labels.to(DEVICE)

        optimizer.zero_grad()
        outputs = jigsaw_model(patches)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * patches.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total_predictions += labels.size(0)
        correct_predictions += (predicted == labels).sum().item()

        progress_bar.set_postfix(loss=loss.item(), acc=f"{(correct_predictions/total_predictions):.2f}")
    
    epoch_acc = correct_predictions / total_predictions
    print(f"Pre-training Эпоха {epoch+1}/{PRETRAIN_EPOCHS} - Точность: {epoch_acc:.4f}")

print("Pre-training завершен.")

# Сохраняем веса обученного backbone
torch.save(jigsaw_model.backbone.state_dict(), 'resnet18_jigsaw_backbone.pth')
print("Веса backbone сохранены в 'resnet18_jigsaw_backbone.pth'")

Начало self-supervised pre-training (Jigsaw)...


Эпоха 1/50:   0%|          | 0/520 [00:00<?, ?it/s]

Pre-training Эпоха 1/50 - Точность: 0.0116


Эпоха 2/50:   0%|          | 0/520 [00:00<?, ?it/s]

Pre-training Эпоха 2/50 - Точность: 0.1312


...

Pre-training Эпоха 48/50 - Точность: 0.9718


Эпоха 49/50:   0%|          | 0/520 [00:00<?, ?it/s]

Pre-training Эпоха 49/50 - Точность: 0.9713


Эпоха 50/50:   0%|          | 0/520 [00:00<?, ?it/s]

Pre-training Эпоха 50/50 - Точность: 0.9733
Pre-training завершен.
Веса backbone сохранены в 'resnet18_jigsaw_backbone.pth'


In [12]:
# Трансформации для основной задачи (остаются такими же)
finetune_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Подготовка размеченных данных (остается такой же)
full_labeled_dataset = datasets.ImageFolder(LABELED_PATH, transform=finetune_transforms)
train_size = int(0.9 * len(full_labeled_dataset))
val_size = len(full_labeled_dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(full_labeled_dataset, [train_size, val_size])
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
class_names = full_labeled_dataset.classes
print(f"Классы для fine-tuning: {class_names}")

# --- Ключевой момент ---
# Инициализируем стандартную модель ResNet-18 для классификации
finetune_model = models.resnet18(num_classes=10)

# Загружаем веса backbone
print("Загрузка весов, полученных на задаче Jigsaw...")
# Создаем модель ResNet без последнего слоя для загрузки весов
backbone_to_load = nn.Sequential(*list(models.resnet18().children())[:-1])
backbone_to_load.load_state_dict(torch.load('resnet18_jigsaw_backbone.pth'))

# Копируем веса в нашу finetune_model
finetune_model.load_state_dict(backbone_to_load.state_dict(), strict=False)
finetune_model = finetune_model.to(DEVICE)

# --- Дальнейшее обучение ---
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(finetune_model.parameters(), lr=LEARNING_RATE_FINETUNE)
best_val_acc = 0.0
best_model_wts = copy.deepcopy(finetune_model.state_dict())

print("Начало supervised fine-tuning...")

for epoch in range(FINETUNE_EPOCHS):
    finetune_model.train()
    # ... (цикл обучения и валидации остается точно таким же, как в предыдущем ноутбуке) ...
    train_correct = 0
    for images, labels in tqdm(train_loader, desc=f"Эпоха {epoch+1} Train"):
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = finetune_model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        _, predicted = torch.max(outputs.data, 1)
        train_correct += (predicted == labels).sum().item()
    
    train_acc = train_correct / len(train_dataset)
    
    finetune_model.eval()
    val_correct = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs = finetune_model(images)
            _, predicted = torch.max(outputs.data, 1)
            val_correct += (predicted == labels).sum().item()
            
    val_acc = val_correct / len(val_dataset)
    print(f"Эпоха {epoch+1}/{FINETUNE_EPOCHS} - Train Acc: {train_acc:.4f}, Val Acc: {val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_model_wts = copy.deepcopy(finetune_model.state_dict())
        torch.save(best_model_wts, 'best_finetuned_jigsaw_model.pth')
        print(f"Новая лучшая модель сохранена с val_acc: {best_val_acc:.4f}")

print(f"Fine-tuning завершен. Лучшая точность на валидации: {best_val_acc:.4f}")

Классы для fine-tuning: ['butterfly', 'cat', 'chicken', 'cow', 'dog', 'elephant', 'horse', 'sheep', 'spider', 'squirrel']
Загрузка весов, полученных на задаче Jigsaw...
Начало supervised fine-tuning...


Эпоха 1 Train:   0%|          | 0/59 [00:00<?, ?it/s]

Эпоха 1/50 - Train Acc: 0.2223, Val Acc: 0.3077
Новая лучшая модель сохранена с val_acc: 0.3077


Эпоха 2 Train:   0%|          | 0/59 [00:00<?, ?it/s]

Эпоха 2/50 - Train Acc: 0.3144, Val Acc: 0.3558
Новая лучшая модель сохранена с val_acc: 0.3558


...

Эпоха 48/50 - Train Acc: 0.9711, Val Acc: 0.6683
Новая лучшая модель сохранена с val_acc: 0.6683


Эпоха 49 Train:   0%|          | 0/59 [00:00<?, ?it/s]

Эпоха 49/50 - Train Acc: 0.9834, Val Acc: 0.6731
Новая лучшая модель сохранена с val_acc: 0.6731


Эпоха 50 Train:   0%|          | 0/59 [00:00<?, ?it/s]

Эпоха 50/50 - Train Acc: 0.9732, Val Acc: 0.6635
Fine-tuning завершен. Лучшая точность на валидации: 0.6731


In [13]:
class TestDataset(Dataset):
    """Dataset for test images."""
    def __init__(self, test_dir, transform=None):
        self.test_dir = test_dir
        self.transform = transform
        self.image_files = sorted([f for f in os.listdir(test_dir) if f.endswith(('.png', '.jpg', '.jpeg'))])

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_path = os.path.join(self.test_dir, self.image_files[idx])
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, self.image_files[idx]

# Use validation transforms for the test set, but without random augmentations
test_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

test_dataset = TestDataset(TEST_PATH, transform=test_transforms)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

# Load the best model for inference
final_model = models.resnet18(num_classes=10)
final_model.load_state_dict(torch.load('best_finetuned_jigsaw_model.pth'))
final_model = final_model.to(DEVICE)
final_model.eval()

predictions = []
image_ids = []

print("Generating predictions on the test set...")
with torch.no_grad():
    for images, fnames in tqdm(test_loader):
        images = images.to(DEVICE)
        outputs = final_model(images)
        _, predicted_indices = torch.max(outputs, 1)

        predictions.extend([class_names[i] for i in predicted_indices.cpu().numpy()])
        image_ids.extend(fnames)

# Create submission DataFrame
submission_df = pd.DataFrame({
    'id': image_ids,
    'class': predictions
})

# Save to CSV
submission_df.to_csv('submission_jigsaw.csv', index=False)

print("Submission file 'submission_jigsaw.csv' created successfully!")
print(submission_df.head())

Generating predictions on the test set...


  0%|          | 0/65 [00:00<?, ?it/s]

Submission file 'submission_jigsaw.csv' created successfully!
         id     class
0     0.jpg       dog
1     1.jpg     horse
2    10.jpg       dog
3   100.jpg    spider
4  1000.jpg  elephant


## Rotations

In [18]:
class RotationDataset(Dataset):
    """Custom Dataset for the rotation pretext task."""
    def __init__(self, image_dir, transform=None):
        self.image_dir = image_dir
        self.transform = transform
        self.image_files = [os.path.join(image_dir, f) for f in os.listdir(image_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_path = self.image_files[idx]
        image = Image.open(img_path).convert('RGB')

        # Apply the base transformations
        if self.transform:
            image = self.transform(image)

        # Apply a random rotation and create the label
        # 0: 0°, 1: 90°, 2: 180°, 3: 270°
        rotation_angle_idx = torch.randint(0, 4, (1,)).item()
        rotated_image = torch.rot90(image, k=rotation_angle_idx, dims=[1, 2])

        return rotated_image, rotation_angle_idx

# Define transformations for the pretext task
pretext_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Create Dataset and DataLoader
pretrain_dataset = RotationDataset(UNLABELED_PATH, transform=pretext_transforms)
pretrain_loader = DataLoader(pretrain_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)

print(f"Found {len(pretrain_dataset)} images for pre-training.")

Found 16611 images for pre-training.


In [19]:
# Load ResNet-18 and modify the classifier for the pretext task
pretext_model = models.resnet18(weights=None) # Start from scratch, no pre-trained weights
num_features = pretext_model.fc.in_features
pretext_model.fc = nn.Linear(num_features, 4) # 4 classes for 4 rotation angles
pretext_model = pretext_model.to(DEVICE)

# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(pretext_model.parameters(), lr=LEARNING_RATE_PRETRAIN)

print("Starting self-supervised pre-training...")

for epoch in range(PRETRAIN_EPOCHS):
    pretext_model.train()
    running_loss = 0.0
    correct_predictions = 0
    total_predictions = 0

    progress_bar = tqdm(pretrain_loader, desc=f"Epoch {epoch+1}/{PRETRAIN_EPOCHS}")

    for images, labels in progress_bar:
        images, labels = images.to(DEVICE), labels.to(DEVICE)

        # Zero the parameter gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = pretext_model(images)
        loss = criterion(outputs, labels)

        # Backward pass and optimize
        loss.backward()
        optimizer.step()

        # Update statistics
        running_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total_predictions += labels.size(0)
        correct_predictions += (predicted == labels).sum().item()

        # Update progress bar
        progress_bar.set_postfix(loss=loss.item(), acc=f"{(correct_predictions/total_predictions):.2f}")

    epoch_loss = running_loss / len(pretrain_loader.dataset)
    epoch_acc = correct_predictions / total_predictions
    print(f"Pre-training Epoch {epoch+1}/{PRETRAIN_EPOCHS} - Loss: {epoch_loss:.4f}, Accuracy: {epoch_acc:.4f}")

print("Finished pre-training.")

# Save the backbone (feature extractor) weights
# We remove the final fully connected layer ('fc') before saving
backbone_weights = copy.deepcopy(pretext_model.state_dict())
keys_to_remove = ["fc.weight", "fc.bias"]
for key in keys_to_remove:
    if key in backbone_weights:
        del backbone_weights[key]

torch.save(backbone_weights, 'resnet18_backbone_pretrained_rotations.pth')
print("Saved pre-trained backbone weights to 'resnet18_backbone_pretrained.pth'")

Starting self-supervised pre-training...


Epoch 1/30:   0%|          | 0/520 [00:00<?, ?it/s]

Pre-training Epoch 1/30 - Loss: 1.2985, Accuracy: 0.3941


Epoch 2/30:   0%|          | 0/520 [00:00<?, ?it/s]

Pre-training Epoch 2/30 - Loss: 1.1725, Accuracy: 0.4859


...

Pre-training Epoch 28/30 - Loss: 0.4357, Accuracy: 0.8358


Epoch 29/30:   0%|          | 0/520 [00:00<?, ?it/s]

Pre-training Epoch 29/30 - Loss: 0.4280, Accuracy: 0.8348


Epoch 30/30:   0%|          | 0/520 [00:00<?, ?it/s]

Pre-training Epoch 30/30 - Loss: 0.4167, Accuracy: 0.8394
Finished pre-training.
Saved pre-trained backbone weights to 'resnet18_backbone_pretrained.pth'


In [20]:
# Transformations for the downstream classification task
finetune_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Create dataset using ImageFolder
full_labeled_dataset = datasets.ImageFolder(LABELED_PATH, transform=finetune_transforms)

# (Recommended) Split labeled data into train and validation sets
train_size = int(0.9 * len(full_labeled_dataset))
val_size = len(full_labeled_dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(full_labeled_dataset, [train_size, val_size])

# Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

# Get class names for submission file later
class_names = full_labeled_dataset.classes
print(f"Found {len(full_labeled_dataset)} labeled images in {len(class_names)} classes.")
print("Classes:", class_names)

Found 2075 labeled images in 10 classes.
Classes: ['butterfly', 'cat', 'chicken', 'cow', 'dog', 'elephant', 'horse', 'sheep', 'spider', 'squirrel']


In [21]:
# Initialize the final classification model
finetune_model = models.resnet18(num_classes=10)

# Load the pre-trained backbone weights
print("Loading pre-trained backbone weights...")
# strict=False allows loading weights into a model with a different final layer
finetune_model.load_state_dict(torch.load('resnet18_backbone_pretrained_rotations.pth'), strict=False)
finetune_model = finetune_model.to(DEVICE)

# Define loss and optimizer for fine-tuning
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(finetune_model.parameters(), lr=LEARNING_RATE_FINETUNE)

best_val_acc = 0.0
best_model_wts = copy.deepcopy(finetune_model.state_dict())

print("Starting supervised fine-tuning...")

for epoch in range(FINETUNE_EPOCHS):
    # Training phase
    finetune_model.train()
    running_loss = 0.0
    correct_predictions = 0
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{FINETUNE_EPOCHS} (Train)")

    for images, labels in progress_bar:
        images, labels = images.to(DEVICE), labels.to(DEVICE)

        optimizer.zero_grad()
        outputs = finetune_model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs.data, 1)
        correct_predictions += (predicted == labels).sum().item()
        progress_bar.set_postfix(loss=loss.item())

    train_acc = correct_predictions / len(train_dataset)

    # Validation phase
    finetune_model.eval()
    val_correct = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs = finetune_model(images)
            _, predicted = torch.max(outputs.data, 1)
            val_correct += (predicted == labels).sum().item()

    val_acc = val_correct / len(val_dataset)
    print(f"Epoch {epoch+1}/{FINETUNE_EPOCHS} - Train Acc: {train_acc:.4f}, Val Acc: {val_acc:.4f}")

    # Save the best model based on validation accuracy
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_model_wts = copy.deepcopy(finetune_model.state_dict())
        torch.save(best_model_wts, 'best_finetuned_model_rotations.pth')
        print(f"New best model saved with validation accuracy: {best_val_acc:.4f}")

print(f"Finished fine-tuning. Best Validation Accuracy: {best_val_acc:.4f}")

Loading pre-trained backbone weights...
Starting supervised fine-tuning...


Epoch 1/30 (Train):   0%|          | 0/59 [00:00<?, ?it/s]

Epoch 1/30 - Train Acc: 0.1628, Val Acc: 0.2163
New best model saved with validation accuracy: 0.2163


Epoch 2/30 (Train):   0%|          | 0/59 [00:00<?, ?it/s]

Epoch 2/30 - Train Acc: 0.2614, Val Acc: 0.3221
New best model saved with validation accuracy: 0.3221


...

Epoch 28/30 - Train Acc: 0.7124, Val Acc: 0.5865


Epoch 29/30 (Train):   0%|          | 0/59 [00:00<?, ?it/s]

Epoch 29/30 - Train Acc: 0.7183, Val Acc: 0.5962


Epoch 30/30 (Train):   0%|          | 0/59 [00:00<?, ?it/s]

Epoch 30/30 - Train Acc: 0.7279, Val Acc: 0.6442
New best model saved with validation accuracy: 0.6442
Finished fine-tuning. Best Validation Accuracy: 0.6442


In [23]:
class TestDataset(Dataset):
    """Dataset for test images."""
    def __init__(self, test_dir, transform=None):
        self.test_dir = test_dir
        self.transform = transform
        self.image_files = sorted([f for f in os.listdir(test_dir) if f.endswith(('.png', '.jpg', '.jpeg'))])

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_path = os.path.join(self.test_dir, self.image_files[idx])
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, self.image_files[idx]

# Use validation transforms for the test set, without random augmentations
test_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

test_dataset = TestDataset(TEST_PATH, transform=test_transforms)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

# Load the best model for inference
final_model = models.resnet18(num_classes=10)
final_model.load_state_dict(torch.load('best_finetuned_model_rotations.pth'))
final_model = final_model.to(DEVICE)
final_model.eval()

predictions = []
image_ids = []

print("Generating predictions on the test set...")
with torch.no_grad():
    for images, fnames in tqdm(test_loader):
        images = images.to(DEVICE)
        outputs = final_model(images)
        _, predicted_indices = torch.max(outputs, 1)

        predictions.extend([class_names[i] for i in predicted_indices.cpu().numpy()])
        image_ids.extend(fnames)

# Create submission DataFrame
submission_df = pd.DataFrame({
    'id': image_ids,
    'class': predictions
})

# Save to CSV
submission_df.to_csv('submission_rotations.csv', index=False)

print("Submission file 'submission_rotations.csv' created successfully!")
print(submission_df.head())

Generating predictions on the test set...


  0%|          | 0/65 [00:00<?, ?it/s]

Submission file 'submission_rotations.csv' created successfully!
         id     class
0     0.jpg       dog
1     1.jpg     horse
2    10.jpg     sheep
3   100.jpg    spider
4  1000.jpg  elephant
